# RAG with Unified Lineage — Working Demo

This notebook demonstrates RudriQ's core thesis end-to-end: a single trace that spans data operations and LLM calls, with cross-domain links automatically attributed.

We use a mock OpenAI client so the notebook runs offline. The instrumentation pattern is identical for real OpenAI calls.

## 1. Set up OpenTelemetry and register RudriQ

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider

from rudriq.processors import RudriQSpanProcessor
from rudriq.linker import register_object_identity, clear_object_registry
from rudriq.processors.linking import record_llm_input, clear_input_registry
from rudriq.storage import get_default_storage

# Reset state so the demo is reproducible.
clear_object_registry()
clear_input_registry()

provider = TracerProvider()
rudriq_proc = RudriQSpanProcessor()
provider.add_span_processor(rudriq_proc)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('rag-demo')

print(f'RudriQ run_id: {rudriq_proc.run_id}')

## 2. Simulate the data pipeline

In production this would be AutoLineage hooks running on real pandas operations. Here we emit equivalent records by hand so the notebook is self-contained.

In [ ]:
from datetime import datetime, timezone
from rudriq.core.schema import NodeKind, TraceNode, TraceEdge, EdgeKind, LinkMethod

storage = get_default_storage()
now = datetime.now(timezone.utc)

# Stage 1: Read a CSV.
read_node = TraceNode(
    node_id='data-read-1',
    kind=NodeKind.DATA_READ, library='pandas', operation='read_csv',
    started_at=now,
    metadata={'path': 'docs.csv', 'rows': 1000},
)
storage.save_node(read_node, run_id=rudriq_proc.run_id)

# Stage 2: Filter to English docs. The output is a real Python object
# that we'll pass to the LLM, so we register its identity with RudriQ.
filtered_docs = ['doc about machine learning', 'doc about retrieval', 'doc about graphs']

filter_node = TraceNode(
    node_id='data-filter-1',
    kind=NodeKind.DATA_TRANSFORM, library='pandas', operation='filter',
    started_at=now,
    metadata={'condition': "lang == 'en'", 'rows_in': 1000, 'rows_out': 3},
)
storage.save_node(filter_node, run_id=rudriq_proc.run_id)
storage.save_edge(
    TraceEdge(parent_id='data-read-1', child_id='data-filter-1',
              kind=EdgeKind.DIRECT),
    run_id=rudriq_proc.run_id,
)

# This is the critical step: register the object passed to the LLM
# so the linker can match it on object identity.
register_object_identity(filtered_docs, 'data-filter-1')
print(f'Registered {len(filtered_docs)} docs with id {id(filtered_docs)} -> data-filter-1')

## 3. Call the LLM with that exact object

We use a span that mimics what OpenLLMetry would emit for `openai.embeddings.create(input=filtered_docs)`. Because the input object's identity matches what we registered, the linker connects the spans automatically.

In [ ]:
with tracer.start_as_current_span('openai.embeddings.create') as span:
    span.set_attribute('gen_ai.system', 'openai')
    span.set_attribute('gen_ai.request.model', 'text-embedding-3-small')
    span.set_attribute('gen_ai.usage.total_tokens', 24)
    
    # Stash the raw Python object so the SpanProcessor can run
    # the linker against it on span end. In v0.1 this happens
    # automatically inside RudriQ's SDK hooks.
    span_id_hex = format(span.context.span_id, '016x')
    record_llm_input(span_id_hex, filtered_docs)
    
    # In real code: response = openai.embeddings.create(input=filtered_docs)
    # We just simulate completion here.
    print(f'LLM call complete (span_id={span_id_hex})')

## 4. Inspect the unified trace

Three nodes (read, filter, embed), two edges (direct read→filter, lineage filter→embed).

In [ ]:
graph = storage.load_run(rudriq_proc.run_id)
print(f'Nodes: {len(graph.nodes)}')
for n in graph.nodes:
    print(f'  {n.node_id:30s} | {n.kind.value:20s} | {n.library}.{n.operation}')

print(f'\nEdges: {len(graph.edges)}')
for e in graph.edges:
    print(f'  {e.parent_id:25s} -> {e.child_id:25s} | {e.kind.value:10s} | {e.link_method.value} (conf={e.confidence:.2f})')

## 5. Walk back from the LLM call to find what fed it

This is the question every existing LLM observability tool can't answer: when this LLM call returned a bad response, which upstream data operation produced its input?

In [ ]:
# Find the LLM node
llm_nodes = [n for n in graph.nodes if n.kind.value.startswith('llm_')]
assert len(llm_nodes) == 1
llm_node = llm_nodes[0]

# Walk lineage parents
lineage_parents = graph.get_lineage_parents(llm_node.node_id)
print(f'The LLM call ({llm_node.node_id}) was fed by:')
for p in lineage_parents:
    print(f'  - {p.operation} ({p.node_id}): {p.metadata}')
    # And recursively up the direct-parent chain
    direct = graph.get_parents(p.node_id)
    for d in direct:
        print(f'      <- {d.operation} ({d.node_id}): {d.metadata}')

## What just happened

RudriQ produced a single graph that spans data lineage and LLM calls. The cross-domain link was automatic — we never told RudriQ "this DataFrame fed this LLM call." Object identity was enough.

In production this is what lets you answer questions like:

- **Why did the response change?** Walk back from the LLM call; the analyzer flags the upstream operation whose output deviated most.
- **Was sensitive data in the prompt?** The lineage chain shows every upstream op; if any of them touched a flagged data source, surface it.
- **What was this AI told?** The signed (in v1.0+) trace artifact records the exact data the model saw.

v0.0.3 is the foundation. The analyzer (Week 3) and audit exporter (Week 4) build on this same graph.